<a href="https://colab.research.google.com/github/rounak-roy-2025/Finance101/blob/rounak-roy-2025-qf/StateSpaceRegime_Nifty50_GMMmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""
NIFTY 50 GMM Market Regime Analysis - GOOGLE COLAB OPTIMIZED VERSION

Features:
1. Uses REAL NIFTY 50 data from Yahoo Finance
2. Multi-colored 3D path based on dominant regime
3. Optimized for Google Colab

Run this cell first to install dependencies:
!pip install -q pandas numpy matplotlib scikit-learn yfinance

Then run the main code below.
"""

# ============================================================================
# INSTALLATION CELL - RUN THIS FIRST IN COLAB
# ============================================================================
# Uncomment and run this in a separate cell:
# !pip install -q pandas numpy matplotlib scikit-learn yfinance

# ============================================================================
# MAIN CODE - RUN THIS AFTER INSTALLATION
# ============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import warnings
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from matplotlib import animation
import yfinance as yf

warnings.filterwarnings('ignore')

print("="*80)
print("NIFTY 50 GMM MARKET REGIME ANALYSIS - REAL DATA VERSION")
print("="*80)
print()

# ============================================================================
# STEP 1: FETCH REAL NIFTY 50 DATA
# ============================================================================
print("STEP 1: Fetching Real NIFTY 50 Data from Yahoo Finance...")
print("-" * 80)

# Download NIFTY 50 data (last 5 trading days, 1-minute intervals)
# Yahoo Finance symbol for NIFTY 50 is ^NSEI
nifty_symbol = "^NSEI"

# Calculate date range - last 5 trading days
end_date = datetime.now()
start_date = end_date - timedelta(days=10)  # Go back 10 days to ensure we get 5 trading days

print(f"Downloading {nifty_symbol} data...")
print(f"Date range: {start_date.date()} to {end_date.date()}")

try:
    # Download 1-minute data
    df = yf.download(nifty_symbol, start=start_date, end=end_date, interval='1m', progress=False)

    if len(df) == 0:
        print("⚠ No 1-minute data available. Using 5-minute data instead...")
        df = yf.download(nifty_symbol, start=start_date, end=end_date, interval='5m', progress=False)

    if len(df) == 0:
        print("⚠ Real-time data unavailable. Using historical daily data...")
        start_date = end_date - timedelta(days=60)
        df = yf.download(nifty_symbol, start=start_date, end=end_date, interval='1d', progress=False)

except Exception as e:
    print(f"⚠ Error fetching data: {e}")
    print("Using last 60 days of daily data as fallback...")
    start_date = end_date - timedelta(days=60)
    df = yf.download(nifty_symbol, start=start_date, end=end_date, interval='1d', progress=False)

# Keep only the last 5 trading days of data
if len(df) > 0:
    # Get unique trading days
    unique_days_list = sorted(list(set([d.date() for d in df.index])))

    # Keep last 5 days
    if len(unique_days_list) > 5:
        last_5_days = unique_days_list[-5:]
        # Filter using list comprehension with reset index
        df = df[[d.date() in last_5_days for d in df.index]]

print(f"✓ Downloaded {len(df)} data points")
print(f"✓ Actual date range: {df.index[0]} to {df.index[-1]}")
print(f"✓ Price range: ₹{float(df['Close'].min()):.2f} to ₹{float(df['Close'].max()):.2f}")
print()

# ============================================================================
# STEP 2: CALCULATE FEATURES
# ============================================================================
print("STEP 2: Calculating Technical Features...")
print("-" * 80)

# Yahoo Finance returns multi-index columns, flatten them
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Calculate Returns
df['Returns'] = df['Close'].pct_change() * 100

# Calculate Rolling Volatility (20-period window for better data)
rolling_window = min(20, len(df) // 10)  # Adaptive window
df['Rolling Volatility'] = df['Returns'].rolling(window=rolling_window).std()

# Generate mock Market Breadth (since real breadth data requires multiple stocks)
np.random.seed(42)
mb_base = 0.5
mb_fluct = np.random.normal(loc=0, scale=0.01, size=len(df))
market_breadth = mb_base + np.cumsum(mb_fluct)
df['Market Breadth'] = np.clip(market_breadth, 0.1, 0.9)

# Generate mock FII Flow
fii_base = 500
fii_fluct = np.random.normal(loc=0.1, scale=5, size=len(df))
df['FII Flow'] = fii_base + np.cumsum(fii_fluct)

# Clean data
df_clean = df.dropna(subset=['Returns', 'Rolling Volatility']).copy()

print(f"✓ Calculated features for {len(df_clean)} valid data points")
print()

# ============================================================================
# STEP 3: FIT GMM
# ============================================================================
print("STEP 3: Fitting Gaussian Mixture Model...")
print("-" * 80)

# Select and scale features
features = ['Returns', 'Rolling Volatility', 'Market Breadth', 'FII Flow']
X = df_clean[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit GMM
gmm = GaussianMixture(n_components=4, random_state=42, max_iter=100)
gmm.fit(X_scaled)

# Get probabilities
probs = gmm.predict_proba(X_scaled)
for i in range(4):
    df_clean[f'GMM_Prob_{i}'] = probs[:, i]

print("✓ GMM fitted with 4 components")
print()

# ============================================================================
# STEP 4: MAP REGIMES
# ============================================================================
print("STEP 4: Mapping Components to Regimes...")
print("-" * 80)

means_original = scaler.inverse_transform(gmm.means_)
components_df = pd.DataFrame(means_original, columns=features)

print("Component Characteristics:")
print(components_df)
print()

# Map components to regimes
regime_map = {
    1: ('Healthy_Expansion', 'Healthy Expansion', 'green'),
    3: ('Stress_Risk_Off', 'Stress / Risk-Off', 'red'),
    0: ('Fragile_Rally', 'Fragile Rally', 'orange'),
    2: ('Repair_Accumulation', 'Repair / Accumulation', 'blue')
}

prob_cols = []
for comp_id, (col_suffix, display_name, color) in regime_map.items():
    col_name = f'P_GMM_{col_suffix}'
    df_clean[col_name] = df_clean[f'GMM_Prob_{comp_id}']
    prob_cols.append(col_name)
    print(f"  Component {comp_id} → {display_name} ({color})")

# Determine dominant regime for each point
def get_dominant_regime(row):
    probs = row[prob_cols].values
    dom_idx = np.argmax(probs)
    return prob_cols[dom_idx].replace('P_GMM_', '')

df_clean['Dominant_Regime'] = df_clean.apply(get_dominant_regime, axis=1)

print()

# ============================================================================
# STEP 5: CREATE ANIMATION WITH MULTI-COLORED PATH
# ============================================================================
print("STEP 5: Creating Animation with Multi-Colored Regime Path...")
print("-" * 80)

# Sample data for Colab performance
sample_rate = max(1, len(df_clean) // 300)  # Aim for ~300 frames
df_sample = df_clean.iloc[::sample_rate].copy()

print(f"Sampling data: {len(df_sample)} frames (every {sample_rate}th point)")

# Create figure
fig = plt.figure(figsize=(16, 8))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122, projection='3d')

# Regime colors
colors = {
    'Healthy_Expansion': 'green',
    'Fragile_Rally': 'orange',
    'Stress_Risk_Off': 'red',
    'Repair_Accumulation': 'blue'
}

# Setup 2D plot
line2d, = ax1.plot([], [], 'b-', lw=2)
ax1.set_title('NIFTY 50 Price Movement', fontsize=13, fontweight='bold')
ax1.set_xlabel('Time', fontsize=10)
ax1.set_ylabel('Price (₹)', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(df_sample.index[0], df_sample.index[-1])
ax1.set_ylim(df_sample['Close'].min() * 0.998, df_sample['Close'].max() * 1.002)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)

# Setup 3D plot - we'll draw line segments instead of one continuous line
ax2.set_title('Regime Space (Color-Coded Path)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Volatility', fontsize=10)
ax2.set_ylabel('Returns (%)', fontsize=10)
ax2.set_zlabel('P(Healthy)', fontsize=10)

ax2.set_xlim(df_sample['Rolling Volatility'].min() * 0.95,
             df_sample['Rolling Volatility'].max() * 1.05)
ax2.set_ylim(df_sample['Returns'].min() * 0.95,
             df_sample['Returns'].max() * 1.05)
ax2.set_zlim(df_sample['P_GMM_Healthy_Expansion'].min() * 0.95,
             df_sample['P_GMM_Healthy_Expansion'].max() * 1.05)

# Current point scatter
scatter3d = ax2.scatter([], [], [], c='k', s=100, edgecolors='w', linewidths=1, zorder=1000)

# Legend
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color='green', label='Healthy Expansion'),
    mpatches.Patch(color='orange', label='Fragile Rally'),
    mpatches.Patch(color='red', label='Stress / Risk-Off'),
    mpatches.Patch(color='blue', label='Repair / Accumulation')
]
ax2.legend(handles=legend_patches, loc='upper left', fontsize=8)

plt.tight_layout()

# Store line segments for 3D path
line_segments = []

# Animation functions
def init():
    line2d.set_data([], [])
    scatter3d._offsets3d = ([], [], [])
    # Clear previous line segments
    for seg in line_segments:
        seg.remove()
    line_segments.clear()
    return [line2d, scatter3d]

def animate(frame):
    # 2D update
    x2d = df_sample.index[:frame+1]
    y2d = df_sample['Close'].iloc[:frame+1]
    line2d.set_data(x2d, y2d)

    # 3D update - draw colored line segments
    if frame > 0:
        # Get data for current and previous point
        x3d_prev = df_sample['Rolling Volatility'].iloc[frame-1]
        y3d_prev = df_sample['Returns'].iloc[frame-1]
        z3d_prev = df_sample['P_GMM_Healthy_Expansion'].iloc[frame-1]

        x3d_curr = df_sample['Rolling Volatility'].iloc[frame]
        y3d_curr = df_sample['Returns'].iloc[frame]
        z3d_curr = df_sample['P_GMM_Healthy_Expansion'].iloc[frame]

        # Get regime color for current segment
        regime_key = df_sample['Dominant_Regime'].iloc[frame]
        segment_color = colors[regime_key]

        # Draw line segment
        line_seg, = ax2.plot([x3d_prev, x3d_curr],
                              [y3d_prev, y3d_curr],
                              [z3d_prev, z3d_curr],
                              color=segment_color, lw=2, alpha=0.8)
        line_segments.append(line_seg)

    # Update current point
    if frame >= 0:
        x3d = df_sample['Rolling Volatility'].iloc[frame]
        y3d = df_sample['Returns'].iloc[frame]
        z3d = df_sample['P_GMM_Healthy_Expansion'].iloc[frame]

        regime_key = df_sample['Dominant_Regime'].iloc[frame]
        point_color = colors[regime_key]

        scatter3d._offsets3d = ([x3d], [y3d], [z3d])
        scatter3d.set_facecolors(point_color)

    ax2.view_init(elev=20, azim=frame * 0.2)

    return [line2d, scatter3d] + line_segments

# Create animation
print("Rendering frames with multi-colored path...")
anim = FuncAnimation(fig, animate, init_func=init,
                     frames=len(df_sample), interval=100,
                     blit=False, repeat=False)  # blit=False for multi-line drawing

# COLAB-SPECIFIC: Display as HTML5 video
print("Converting to HTML5 video...")
html_video = HTML(anim.to_html5_video())

plt.close(fig)

print("✓ Animation created successfully with color-coded regime path")
print()
print("="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print()
print("Features:")
print("  ✓ Real NIFTY 50 data from Yahoo Finance")
print("  ✓ Multi-colored 3D path showing regime transitions")
print("  ✓ Green = Healthy Expansion")
print("  ✓ Orange = Fragile Rally")
print("  ✓ Red = Stress / Risk-Off")
print("  ✓ Blue = Repair / Accumulation")
print()

# Display the animation
html_video

NIFTY 50 GMM MARKET REGIME ANALYSIS - REAL DATA VERSION

STEP 1: Fetching Real NIFTY 50 Data from Yahoo Finance...
--------------------------------------------------------------------------------
Date range: 2026-01-15 to 2026-01-25


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['^NSEI']: YFPricesMissingError('possibly delisted; no price data found  (1m 2026-01-15 09:28:34.377533 -> 2026-01-25 09:28:34.377533) (Yahoo error = "1m data not available for startTime=1768449514 and endTime=1769313514. Only 8 days worth of 1m granularity data are allowed to be fetched per request.")')


⚠ No 1-minute data available. Using 5-minute data instead...
✓ Downloaded 375 data points
✓ Actual date range: 2026-01-19 03:45:00+00:00 to 2026-01-23 09:55:00+00:00
✓ Price range: ₹24969.45 to ₹25628.95

STEP 2: Calculating Technical Features...
--------------------------------------------------------------------------------
✓ Calculated features for 355 valid data points

STEP 3: Fitting Gaussian Mixture Model...
--------------------------------------------------------------------------------
✓ GMM fitted with 4 components

STEP 4: Mapping Components to Regimes...
--------------------------------------------------------------------------------
Component Characteristics:
    Returns  Rolling Volatility  Market Breadth    FII Flow
0 -0.001068            0.101116        0.420617  439.721482
1 -0.005629            0.078407        0.515278  457.842727
2 -0.009282            0.053516        0.416878  494.791105
3  0.020521            0.155684        0.492966  458.285993

  Component 1 → He